![image](https://raw.githubusercontent.com/ambideXtrous9/Finetune-Qwen3-using-Unsloth/refs/heads/main/Experimental/Qwen3.5.jpg)


In [1]:
!nvidia-smi

Sat Mar 14 13:28:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   61C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## **🦥 Install Packages**

In [2]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
!uv pip install -qqq "numpy==1.26.4" --force-reinstall
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps tokenizers trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
# causal_conv1d is supported only on torch==2.8.0. If you have newer torch versions, please wait 10 minutes!
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0 xformers==0.0.32.post2
!uv pip install -qq -U evaluate rouge_score

In [3]:
import unsloth
from unsloth import FastLanguageModel
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
import os
os.environ["UNSLOTH_RETURN_LOGITS"] = "1"

In [5]:
max_seq_length = 1024

## **🦥 Loading Qwen3.5-0.8B Model**

In [6]:
# Load the base model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3.5-0.8B",
    max_seq_length = max_seq_length,   # Define context length
    load_in_4bit = True,     # 4bit uses much less memory
    load_in_8bit = False,    # A bit more accurate, uses 2x memory
    full_finetuning = False, # We have full finetuning now!
    # token = "hf_...",      # Add your token if using a gated model
)

==((====))==  Unsloth 2026.3.4: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

In [7]:
# Add LoRA adapters to the model
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,           # LoRA rank (higher rank = more parameters, potentially better fit but more memory)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", # Target attention and MLP layers
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,  # Scaling factor (often set to r or 2*r)
    lora_dropout = 0, # Dropout probability for LoRA layers
    bias = "none",    # Fine-tuning bias terms ('none' is often optimal)
    # Use Unsloth's gradient checkpointing for memory saving
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False, # Rank Stable LoRA (optional)
    loftq_config = None, # LoftQ initialization (optional)
)

Unsloth: Making `model.base_model.model.model.language_model` require gradients


## **🦥 Loading Reasoning and Non-Reasoning Dataset**

In [8]:
from datasets import load_dataset
reasoning_dataset = load_dataset("unsloth/OpenMathReasoning-mini", split = "cot")
non_reasoning_dataset = load_dataset("mlabonne/FineTome-100k", split = "train")

In [9]:
print("Reasoning Dataset Example Row:")
print(reasoning_dataset.shape)
print("\nNon-Reasoning Dataset Example Row (raw):")
print(non_reasoning_dataset.shape)

Reasoning Dataset Example Row:
(19252, 8)

Non-Reasoning Dataset Example Row (raw):
(100000, 3)


In [10]:
reasoning_dataset.column_names

['expected_answer',
 'problem_type',
 'problem_source',
 'generation_model',
 'pass_rate_72b_tir',
 'problem',
 'generated_solution',
 'inference_mode']

## **🦥 Converting Dataset to Compatible Format**

In [11]:
def generate_reasoning_conversation(examples):
    problems  = examples["problem"]
    # The 'generated_solution' contains the Chain-of-Thought reasoning
    solutions = examples["generated_solution"]
    conversations = []
    for problem, solution in zip(problems, solutions):
        conversations.append([
            {"role" : "user",      "content" : problem},
            # The solution here includes the <think>...</think> block already formatted
            {"role" : "assistant", "content" : solution},
        ])
    return { "conversations": conversations, }


In [12]:
# Step 1: Generate conversations
mapped_reasoning = reasoning_dataset.map(
    generate_reasoning_conversation, 
    batched=True, 
    num_proc=1,
    remove_columns=reasoning_dataset.column_names  # Remove original columns
)

# Step 2: Apply chat template
reasoning_formatted_texts = []
for conv in mapped_reasoning["conversations"]:
    formatted = tokenizer.apply_chat_template(conv, tokenize=False)
    reasoning_formatted_texts.append(formatted)

print(f"\nTotal formatted texts: {len(reasoning_formatted_texts)}")


Total formatted texts: 19252


In [13]:
from unsloth.chat_templates import standardize_sharegpt

# Standardize the ShareGPT format first (if applicable)
standardized_non_reasoning = standardize_sharegpt(non_reasoning_dataset)

# Apply the chat template to each conversation
non_reasoning_formatted_texts = [
    tokenizer.apply_chat_template(conv, tokenize=False)
    for conv in standardized_non_reasoning["conversations"]
]

print(f"\nTotal formatted Non-Reasoning texts: {len(non_reasoning_formatted_texts)}")


Total formatted Non-Reasoning texts: 100000


## **🦥 Picking 1000 Examples as Dataset**

In [14]:
import pandas as pd
from datasets import Dataset

In [15]:
import pandas as pd

# Assume these are defined already
# reasoning_formatted_texts: list or iterable with 19,252 items
# non_reasoning_formatted_texts: list or iterable with at least 15,000 items

chat_percentage = 0.75  # aim for 75% chat (reasoning) data

reasoning_series = pd.Series(reasoning_formatted_texts)
non_reasoning_series = pd.Series(non_reasoning_formatted_texts)

num_reasoning = 500
num_non_reasoning = 1000

# Ensure we don't oversample from the available data
num_reasoning = min(num_reasoning, len(reasoning_series))
num_non_reasoning = min(num_non_reasoning, len(non_reasoning_series))

# Sample
reasoning_sample = reasoning_series.sample(n=num_reasoning, random_state=42)
non_reasoning_sample = non_reasoning_series.sample(n=num_non_reasoning, random_state=42)

print(f"Using {len(reasoning_sample)} reasoning samples.")
print(f"Sampling {len(non_reasoning_sample)} non-reasoning samples.")


Using 500 reasoning samples.
Sampling 1000 non-reasoning samples.


In [16]:
# # Define desired chat data percentage
# chat_percentage = 0.75 # Aim for 75% chat data
# # Convert to Pandas Series for easier sampling
# reasoning_series = pd.Series(reasoning_formatted_texts)
# non_reasoning_series = pd.Series(non_reasoning_formatted_texts)
# # Sample non-reasoning data based on the desired ratio relative to reasoning data
# # Calculate how many non-reasoning samples we need
# num_non_reasoning_samples = int(len(reasoning_series) * (chat_percentage / (1.0 - chat_percentage)))
# # Ensure we don't request more samples than available
# num_non_reasoning_samples = min(num_non_reasoning_samples, len(non_reasoning_series))

# print(f"Using {len(reasoning_series)} reasoning samples.")
# print(f"Sampling {num_non_reasoning_samples} non-reasoning samples.")

In [17]:
non_reasoning_subset = non_reasoning_series.sample(
    n = len(non_reasoning_sample),
    random_state = 2407, # for reproducibility
)

# Combine the datasets
combined_series = pd.concat([reasoning_series, non_reasoning_subset])
combined_series.name = "text" # The SFTTrainer expects this column name

# Convert back to Hugging Face Dataset and shuffle
combined_dataset = Dataset.from_pandas(pd.DataFrame(combined_series))


combined_dataset = combined_dataset.shuffle(seed = 3407)

# Take the first 1000 rows as a new Dataset
small_dataset = combined_dataset.select(range(1000))

print(f"Small dataset has {len(small_dataset)} rows.")

print(f"\nFinal Combined Dataset size: {len(combined_dataset)}")
#print("Example entry from combined dataset:")
#print(combined_dataset[0]['text'])

Small dataset has 1000 rows.

Final Combined Dataset size: 20252


## **🦥 Train-Test Split**

In [18]:
# Split into 90% train and 10% validation
split_dataset = small_dataset.train_test_split(test_size=0.2, seed=3407)

# Access the train and validation sets
train_dataset = split_dataset["train"]
valid_dataset = split_dataset["test"]


In [19]:
def tokenize_function(examples):
    return tokenizer(
        text=examples["text"],   # ← must be a keyword argument
        images=None,             # ← explicitly no images
        videos=None,             # ← explicitly no videos
        truncation=True,
        padding="max_length",
        max_length=max_seq_length,
    )

In [20]:
train_dataset = train_dataset.map(tokenize_function, batched=True, num_proc=1, remove_columns=["text"])
valid_dataset = valid_dataset.map(tokenize_function, batched=True, num_proc=1, remove_columns=["text"])

Map (num_proc=1):   0%|          | 0/800 [00:00<?, ? examples/s]

Map (num_proc=1):   0%|          | 0/200 [00:00<?, ? examples/s]

## **🦥 Evaluation Metrices : Rouge, BLEU**

In [21]:
import numpy as np
from transformers import EvalPrediction
import evaluate

# Load metrics
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")


In [22]:

def preprocess_logits_for_metrics(logits, labels):
    """Returns predicted token IDs (argmax) for metrics calculation"""
    if isinstance(logits, tuple):
        logits = logits[0]  # Unpack if needed
    return logits.argmax(dim=-1)

def compute_metrics(eval_preds: EvalPrediction):
    """Compute ROUGE, BLEU, AND token-level accuracy"""
    preds, labels = eval_preds
    
    # --- Token-Level Accuracy Calculation ---
    # Flatten all predictions/labels (ignore padding tokens)
    mask = labels != -100  # Only compare non-ignored tokens
    preds_flat = preds[mask].flatten()
    labels_flat = labels[mask].flatten()
    
    accuracy = (preds_flat == labels_flat).mean()
    
    # --- Text Generation Metrics ---
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Post-process text
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]
    
    rouge_results = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )
    bleu_results = bleu.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )
    
    return {
        "accuracy": float(accuracy),  # Token-level exact match
        "rouge1": rouge_results["rouge1"],
        "rouge2": rouge_results["rouge2"],
        "rougeL": rouge_results["rougeL"],
        "bleu": bleu_results["bleu"],
    }

In [23]:
from trl import SFTTrainer, SFTConfig

## **🦥 SFT Training Args**

In [24]:
sftconfig = SFTConfig(
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 8, # Effective batch size = 2 * 4 = 8
        warmup_steps = 1,
        max_steps = 5,                 # Short run for demonstration; set to None for full epochs
        # num_train_epochs = 1,         # Alternatively, train for 1 full epoch
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(), # Use bf16 if available, else fp16
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",           # Use 8-bit AdamW optimizer
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        per_device_eval_batch_size=8,
        seed = 3407,
        dataloader_pin_memory=True, #fast gpu data transfer
        output_dir = "outputs",         # Directory to save checkpoints
        report_to = "none",             # Disable external reporting (like WandB) for this example
        eval_strategy="steps",  # Evaluate during training
        eval_steps=5,                 # Evaluate every 5 steps
        fp16_full_eval = True,
        eval_accumulation_steps=1,
        load_best_model_at_end=True, # Load best model based on evaluation metric
        metric_for_best_model="eval_loss",  # You can also use "eval_loss"
        greater_is_better=False,           # For accuracy, higher is better
        dataset_num_proc=1

    )

## **🦥 Trainer**

In [25]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    args=sftconfig,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    compute_metrics=compute_metrics,
)

Unsloth: Switching to float32 training since model cannot work with float16


## **🦥 Compute loss on Response Only**

In [26]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",       # matches your chat template
    response_part    = "<|im_start|>assistant\n",  # aligns with assistant replies
)


Map (num_proc=6):   0%|          | 0/800 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/800 [00:00<?, ? examples/s]

Map (num_proc=6):   0%|          | 0/200 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/200 [00:00<?, ? examples/s]

In [27]:
# Start training
print("Starting training...")
trainer_stats = trainer.train()
print("Training finished.")


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 800 | Num Epochs = 1 | Total steps = 5
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 8 x 1) = 64
 "-____-"     Trainable parameters = 6,389,760 of 859,375,680 (0.74% trained)


Step,Training Loss,Validation Loss,Accuracy,Rouge1,Rouge2,Rougel,Bleu
5,0.695940,0.670615,0.006989,0.826085,0.570632,0.709113,0.571730


Unsloth: Not an error, but Qwen3_5ForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Training finished.


In [28]:
# print training stats
print(trainer_stats)

TrainOutput(global_step=5, training_loss=0.7015538692474366, metrics={'train_runtime': 670.4484, 'train_samples_per_second': 0.477, 'train_steps_per_second': 0.007, 'total_flos': 1186188220170240.0, 'train_loss': 0.7015538692474366, 'epoch': 0.4})


## **🦥 Streaming Inference**

In [31]:
from transformers import TextStreamer

messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Solve (x + 2)^2 = 0."}  # ← list of dicts, not a string
        ]
    }
]

In [32]:
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    enable_thinking=False,
    return_tensors="pt",
    return_dict=True,
).to("cuda")

Keyword argument `enable_thinking` is not a valid argument for this processor and will be ignored.


In [33]:
streamer_no_think = TextStreamer(tokenizer, skip_prompt=True)

_ = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.7,
    top_p=0.8,
    top_k=20,
    streamer=streamer_no_think,
    eos_token_id=tokenizer.eos_token_id,
)
print("\n-----------------------------")

To solve the equation $(x + 2)^2 = 0$, we can follow these steps:

1.  **Expand the equation**:
    The equation is in the form of a perfect square, $(a + b)^2 = a^2 + 2ab + b^2$.
    Here, $a = x$ and $b = 2$.

2.  **Substitute the values**:
    $$x^2 + 2(x)(2) + 2^2 = 0$$
    $$x^2 + 4x + 4 = 0$$

3.  **Factor the equation**:
    Notice that the expression is a perfect square trinomial:
    $$(x + 2)^2 = 0$$

4.  **Solve for $x$**:
    Taking the square root of both sides:
    $$x + 2 = 0$$
    $$x = -2$$

Alternatively, you can simply set the term inside the square equal to zero:
$$x + 2 = 0$$
$$x = -2$$

**Answer:**
$$x = -2$$<|im_end|>

-----------------------------


## **🦥 Thinking Inference**

In [40]:
messages_think = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "/think"}]
    },
    {
        "role": "user",
        "content": [{"type": "text", "text": "Solve (x + 2)^2 = 0."}]
    }
]

In [41]:
inputs_think = tokenizer.apply_chat_template(
    messages_think,
    tokenize=True,
    enable_thinking=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to("cuda")


In [42]:
streamer_think = TextStreamer(tokenizer, skip_prompt=True)

_ = model.generate(
    **inputs_think,
    max_new_tokens=1024,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    streamer=streamer_think,
    eos_token_id=tokenizer.eos_token_id,
)
print("\n-----------------------------")

Okay, so I need to solve the equation (x + 2)² = 0. Hmm, let me think. I remember that squaring a number gives a positive result, right? So if I have (x + 2) squared equals zero, that means the expression inside the square must be zero. Because if you take the square root of zero, you get zero. So, x + 2 = 0. Then, to solve for x, I just subtract 2 from both sides. That gives x = -2. Yeah, that seems straightforward. Let me double-check. If I plug x = -2 back into the original equation, (−2 + 2)² equals zero squared, which is zero. Yep, that works. Alright, I think that's the solution.
</think>

To solve the equation $(x + 2)^2 = 0$, we can follow these steps:

1. **Set the expression inside the square equal to zero**:
   $$x + 2 = 0$$

2. **Solve for $x$ by subtracting 2 from both sides**:
   $$x = -2$$

Thus, the solution to the equation $(x + 2)^2 = 0$ is:
$$\boxed{-2}$$<|im_end|>

-----------------------------


In [39]:
# Save LoRA adapters locally
model.save_pretrained("qwen3_0.8b_reasoning_chat_lora")
tokenizer.save_pretrained("qwen3_0.8b_reasoning_chat_lora")

print("LoRA adapters saved locally to 'qwen3_0.8b_reasoning_chat_lora'")

# Optional: Push to Hugging Face Hub
# model.push_to_hub("your_username/qwen3_14b_reasoning_chat_lora", token="YOUR_HF_TOKEN")
# tokenizer.push_to_hub("your_username/qwen3_14b_reasoning_chat_lora", token="YOUR_HF_TOKEN")

# To load these adapters later:
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name = "qwen3_14b_reasoning_chat_lora", # Path to saved adapters
#     load_in_4bit = True,
# )

LoRA adapters saved locally to 'qwen3_0.8b_reasoning_chat_lora'
